## Import Library

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

import os, json, pickle, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.utils import compute_class_weight
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import (
    Embedding, LSTM, GRU, Dense, Dropout,
    Bidirectional, GlobalMaxPool1D, Conv1D,
    MaxPooling1D, BatchNormalization, Flatten, GlobalAveragePooling1D,
)
from tensorflow.keras.callbacks import TensorBoard
import datetime
from collections import Counter
# Untuk custom layer nanti:
from tensorflow.keras.layers import Layer


In [ ]:
# Inisiasi variabel Global
RANDOM_SEED   = 42
NUM_WORDS     = 10000
MAX_LENGTH    = 200
EMBED_DIM     = 128
EPOCH         = 50
BATCH_SIZE    = 32
OUTPUT_DIR = './models'
LOGS_DIR   = './logs'
DATSET_PATH = 'D:\\Kuliah\\DICODING\\Capstone\\HealMate_AI\\AI\\Emotion Dataset Merged.xlsx'
# AI\Emotion Dataset Merged.xlsx
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)
LOG_DIR = os.path.join(LOGS_DIR, datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))


## Load Dataset

In [ ]:
df =pd.read_excel(DATSET_PATH)
print(f'Shape : {df.shape}')
df.head()

## EDA

In [ ]:
df.info()
print('\nMissing values:')
print(df.isnull().sum())
print(f'\nDuplikat : {df.duplicated().sum()}')
df.describe()

In [ ]:

#Distribusi panjang teks
df['input_length'] = df['input_clean'].str.split().str.len()
df['label_length'] = df['label_clean'].str.split().str.len()
print(df[['input_length', 'label_length']].describe())

#Cek duplikat input (input sama, label berbeda)
print(f"Input unik: {df['input_clean'].nunique()} dari {len(df)} total")

#Cek label noise (confidence rendah)
print(df[df['emotion_confidence'] < 0.5]['predicted_emotion'].value_counts())


In [ ]:
# ── Distribusi Kelas ─────────────────────────────────────────
label_counts = df['predicted_emotion'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(label_counts.index, label_counts.values,
            color=['#e74c3c', '#3498db', '#2ecc71'])
axes[0].set_title('Distribusi Kelas Emosi')
axes[0].set_xlabel('Emosi')
axes[0].set_ylabel('Jumlah')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(label_counts.values, labels=label_counts.index,
            autopct='%1.1f%%', colors=['#e74c3c', '#3498db', '#2ecc71'],
            startangle=90)
axes[1].set_title('Proporsi Kelas')

plt.tight_layout()
plt.show()
print(label_counts)

## Split Data

In [ ]:
X = df['input_clean'].fillna('').astype(str)
y = df['predicted_emotion']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print('Label encoding:')
for i, cls in enumerate(le.classes_):
    print(f'  {cls:12s} → {i}')

with open(os.path.join(OUTPUT_DIR, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(le, f)
print(" Label encoder disimpan.")

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded, test_size=0.2, random_state=RANDOM_SEED, stratify=y_encoded
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_SEED, stratify=y_temp
)

print(f'\nData split:')
print(f'  Train set : {len(X_train):,} samples ({len(X_train)/len(X)*100:.1f}%)')
print(f'  Val set   : {len(X_val):,} samples ({len(X_val)/len(X)*100:.1f}%)')
print(f'  Test set  : {len(X_test):,} samples ({len(X_test)/len(X)*100:.1f}%)')

print("Distribusi train:", Counter(y_train))
print("Distribusi val:  ", Counter(y_val))
print("Distribusi test: ", Counter(y_test))

# Simpan index train untuk mapping retrieval
train_indices = X_train.index.tolist()
print(f'\n Index train disimpan untuk mapping retrieval ({len(train_indices)} baris)')

## Tokenisasi dan Padding

In [ ]:
tokenizer = Tokenizer(num_words=NUM_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

vocab_size = min(len(tokenizer.word_index) + 1, NUM_WORDS)
print(f'Vocab size (aktual) : {len(tokenizer.word_index):,}')
print(f'Vocab size (capped) : {vocab_size:,}')

def texts_to_padded(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LENGTH, padding='post', truncating='post')

X_train_seq = texts_to_padded(X_train)
X_val_seq   = texts_to_padded(X_val)
X_test_seq  = texts_to_padded(X_test)

print(f'\nShape setelah padding:')
print(f'  Train : {X_train_seq.shape}')
print(f'  Val   : {X_val_seq.shape}')
print(f'  Test  : {X_test_seq.shape}')

with open(os.path.join(OUTPUT_DIR, 'tokenizer.pkl'), 'wb') as f:
    pickle.dump(tokenizer, f)
print(" Tokenizer disimpan.")

## Ekstraksi Fitur

### TF-IDF

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.8, ngram_range=(1, 2))
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf   = tfidf_vectorizer.transform(X_val)
X_test_tfidf  = tfidf_vectorizer.transform(X_test)
print(f'TF-IDF matrix shape: {X_train_tfidf.shape}')

smote_tfidf = SMOTE(random_state=RANDOM_SEED)
X_train_tfidf_smote, y_train_tfidf_smote = smote_tfidf.fit_resample(X_train_tfidf, y_train)

print('\nSMOTE TF-IDF — distribusi setelah resampling:')
unique, counts = np.unique(y_train_tfidf_smote, return_counts=True)
for cls, cnt in zip(le.classes_, counts):
    print(f'  {cls:12s}: {cnt}')
print(f'  Total: {len(y_train_tfidf_smote):,}')

### CountVectorizer

In [ ]:
count_vectorizer = CountVectorizer(max_features=5000, min_df=5, max_df=0.8, ngram_range=(1, 2))
X_train_count = count_vectorizer.fit_transform(X_train)
X_val_count   = count_vectorizer.transform(X_val)
X_test_count  = count_vectorizer.transform(X_test)
print(f'Count matrix shape: {X_train_count.shape}')

smote_count = SMOTE(random_state=RANDOM_SEED)
X_train_count_smote, y_train_count_smote = smote_count.fit_resample(X_train_count, y_train)

print('\nSMOTE Count — distribusi setelah resampling:')
unique, counts = np.unique(y_train_count_smote, return_counts=True)
for cls, cnt in zip(le.classes_, counts):
    print(f'  {cls:12s}: {cnt}')
print(f'  Total: {len(y_train_count_smote):,}')

## Modeling

### Custom component


In [ ]:
class customEarlyStopping(tf.keras.callbacks.Callback):
    def __init__(self, patience=3, min_delta=0.001):
        super(customEarlyStopping, self).__init__()
        self.patience = patience
        self.min_delta = min_delta
        self.wait = 0
        self.best_loss = np.Inf

    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get('val_loss')
        if current_loss is None:
            return

        if current_loss < self.best_loss - self.min_delta:
            self.best_loss = current_loss
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"\nEarly stopping at epoch {epoch+1}")
                self.model.stop_training = True


class stopTrainingAtAccuracy(tf.keras.callbacks.Callback):
    def __init__(self, target_acc=0.95):
        super(stopTrainingAtAccuracy, self).__init__()
        self.target_acc = target_acc

    def on_epoch_end(self, epoch, logs=None):
        acc = logs.get('accuracy')
        if acc is not None and acc >= self.target_acc:
            print(f"\nTarget accuracy {self.target_acc:.2f} reached at epoch {epoch+1}")
            self.model.stop_training = True

class CustomModelCheckpoint(tf.keras.callbacks.Callback):
    def __init__(self, filepath, monitor='val_loss', save_best_only=True):
        super(CustomModelCheckpoint, self).__init__()
        self.filepath = filepath
        self.monitor = monitor
        self.save_best_only = save_best_only
        self.best = np.Inf if 'loss' in monitor else -np.Inf

    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is None:
            return

        if (self.save_best_only and 
            ((current < self.best and 'loss' in self.monitor) or 
             (current > self.best and 'acc' in self.monitor))):
            print(f"\nSaving best model to {self.filepath} (monitor: {self.monitor}={current:.4f})")
            self.model.save(self.filepath)
            self.best = current


class CustomTensorBoard(tf.keras.callbacks.TensorBoard):
    def __init__(self, log_dir=LOG_DIR, **kwargs):
        super(CustomTensorBoard, self).__init__(log_dir=log_dir, **kwargs)

    def on_epoch_end(self, epoch, logs=None):
        super().on_epoch_end(epoch, logs)
        if logs is not None:
            print(f"Epoch {epoch+1} — loss: {logs.get('loss'):.4f}, val_loss: {logs.get('val_loss'):.4f}, acc: {logs.get('accuracy'):.4f}, val_acc: {logs.get('val_accuracy'):.4f}")


class customDetecOverfitting(tf.keras.callbacks.Callback):
    def __init__(self, patience=3, min_delta=0.001):
        super(customDetecOverfitting, self).__init__()
        self.patience = patience
        self.min_delta = min_delta
        self.wait = 0
        self.best_loss = np.Inf

    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get('val_loss')
        if current_loss is None:
            return

        if current_loss < self.best_loss - self.min_delta:
            self.best_loss = current_loss
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"\nPotential overfitting detected at epoch {epoch+1} (val_loss: {current_loss:.4f} did not improve for {self.patience} epochs)")

class TrainingSummaryLogger(tf.keras.callbacks.Callback):
    def __init__(self, save_path=None):
        super().__init__()
        self.save_path = save_path
 
    def on_train_begin(self, logs=None):
        self.start_time = time.time()
        total_batches = self.params.get("steps", "?") 
        print(f"\nTotal batch per epoch: {total_batches}\n")
 
    def on_train_end(self, logs=None):
        total_time = time.time() - self.start_time
        avg_loss = logs.get("loss", "N/A")
 
        print(f"\nWaktu training {total_time:.2f} detik.")
        print(f"Rata-rata Loss Akhir: {avg_loss}\n")
 
        # Simpan model jika save_path diberikan
        if self.save_path:
            self.model.save(self.save_path)
            print(f"Model disimpan ke: {self.save_path}\n")


class CustomLossFunction(tf.keras.losses.Loss):
    def __init__(self, mse_weight=0.01, **kwargs):
        super().__init__(**kwargs)
        self.mse_weight = mse_weight

    def call(self, y_true, y_pred):

        ce_loss = tf.keras.losses.sparse_categorical_crossentropy(
            y_true, y_pred
        )
        y_true_onehot = tf.one_hot(
            tf.cast(y_true, tf.int32),
            depth=tf.shape(y_pred)[-1]
        )
        y_pred_prob = tf.nn.softmax(y_pred)
        mse_loss = tf.reduce_mean(tf.square(y_true_onehot - y_pred_prob))
    
    def get_config(self):
        cfg = super().get_config()
        cfg.update({'mse_weight': self.mse_weight})
        return cfg

        return ce_loss + self.mse_weight * mse_loss
    
class customLearningRateScheduler(tf.keras.callbacks.Callback):
    def __init__(self, initial_lr=0.001, decay_factor=0.5, step_size=5):
        super(customLearningRateScheduler, self).__init__()
        self.initial_lr = initial_lr
        self.decay_factor = decay_factor
        self.step_size = step_size

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.step_size == 0:
            new_lr = self.initial_lr * (self.decay_factor ** ((epoch + 1) // self.step_size))
            tf.keras.backend.set_value(self.model.optimizer.lr, new_lr)
            print(f"\nEpoch {epoch+1}: Learning rate adjusted to {new_lr:.6f}")

class CustomDenseLayer(tf.keras.layers.Layer):
    def __init__(self, units=32, activation='relu', dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = tf.keras.activations.get(activation)
        self.dropout_rate = dropout_rate

    def build(self, input_shape):
        self.W = self.add_weight(
            name='kernel',
            shape=(input_shape[-1], self.units),
            initializer='glorot_uniform',
            trainable=True
        )
        self.b = self.add_weight(
            name='bias',
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )
        self.dropout = tf.keras.layers.Dropout(self.dropout_rate)
        super().build(input_shape)

    def call(self, inputs, training=False):
        x = tf.matmul(inputs, self.W) + self.b
        x = self.activation(x)
        x = self.dropout(x, training=training)
        return x

    def get_config(self):  
        cfg = super().get_config()
        cfg.update({'units': self.units,
                    'dropout_rate': self.dropout_rate})
        return cfg
    
class CustomEmbeddingLayer(tf.keras.layers.Layer):
    def __init__(self, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

    def build(self, input_shape):
        self.embedding_matrix = self.add_weight(
            name='embedding_matrix',
            shape=(self.vocab_size, self.embed_dim),
            initializer=tf.keras.initializers.TruncatedNormal(stddev=0.02),
            trainable=True
        )
        super().build(input_shape)

    def call(self, token_ids):
        x = tf.nn.embedding_lookup(self.embedding_matrix, token_ids)
        mask = tf.cast(tf.not_equal(token_ids, 0), tf.float32)
        mask = tf.expand_dims(mask, -1)
        return x * mask

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'vocab_size': self.vocab_size, 'embed_dim': self.embed_dim})
        return cfg
    
    
class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        ce = tf.keras.losses.sparse_categorical_crossentropy(
            y_true, y_pred, from_logits=True
        )
        pt = tf.exp(-ce)
        focal = self.alpha * tf.pow(1.0 - pt, self.gamma) * ce
        return tf.reduce_mean(focal)
    
class OverfittingDetector(tf.keras.callbacks.Callback):
    def __init__(self, threshold=0.15):
        super().__init__()
        self.threshold = threshold  

    def on_epoch_end(self, epoch, logs=None):
        train_loss = logs.get('loss', 0)
        val_loss   = logs.get('val_loss', 0)
        gap = val_loss - train_loss

        if gap > self.threshold:
            print(f"\n⚠ Overfitting terdeteksi epoch {epoch+1}: "
                  f"gap={gap:.4f} (train={train_loss:.4f}, val={val_loss:.4f})")
    

## Evaluasi dan Perbandingan

## Simpan Model